# Paso 1 -- Enfoque del agente PPO, y entrenamiento final

Conecta un agente de RL (PPO -- Proximal Policy Optimization, no DQN: el DQN
de Stable-Baselines3 no soporta `MultiDiscrete` de forma nativa, PPO si)
sobre `EntornoDemandaAleatoria` (`modelo_rl/src/entrenamiento.py`), que
regenera la demanda en cada `reset()` en vez de reusar una tabla fija --
necesario para que el agente no memorice una unica realizacion de demanda.

**Instancia de entrenamiento:** los parametros de `escalon_1` (franja
manana, 2 barcos, pocos grupos) -- la instancia chica, para poder iterar
rapido en la secuencia de experimentos (`02_secuencia_experimentos_reward.ipynb`)
antes de escalar. Los hiperparametros de PPO (normalizacion, entropia,
frecuencia de actualizacion) viven en `simulador/config/instance.yaml` ->
`agente.hiperparametros`, COMPARTIDOS entre los experimentos y esta corrida
final -- ya fueron validados en una ronda de diagnostico anterior, no se
vuelven a tocar aca. La recompensa (`agente.entrenamiento.recompensa_overrides`)
si es especifica de esta corrida: es el combo GANADOR de la secuencia de
experimentos (ver el notebook 02) -- correr este notebook 01 DESPUES del 02,
no antes.

**Orden:** confirmar demanda fresca por reset -> `check_env` (chequeo
estandar de SB3, antes de entrenar) -> entrenar (unica corrida LARGA de
todo el ciclo) -> guardar el modelo y graficar la curva de recompensa por
episodio.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

BASE_DIR = Path.cwd().parent
SIMULADOR_DIR = (BASE_DIR / "../simulador").resolve()
BB_DIR = (BASE_DIR / "../bergen-boats").resolve()
DEMAND_DIR = (BASE_DIR / "../demand").resolve()
sys.path.insert(0, str(DEMAND_DIR / "src"))
sys.path.insert(0, str(SIMULADOR_DIR / "src"))
sys.path.insert(0, str(BASE_DIR / "src"))

import masas as demand_masas
import llegadas as demand_llegadas
from entrenamiento import EntornoDemandaAleatoria

cfg_sim = yaml.safe_load(open(SIMULADOR_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_bb = yaml.safe_load(open(BB_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_demand = demand_masas.cargar_config(DEMAND_DIR / "config" / "instance.yaml")

resumen_masas = pd.read_csv(DEMAND_DIR / "output" / "masas_por_nodo.csv", index_col="id")
intensidad_od = pd.read_csv(DEMAND_DIR / "output" / "matriz_intensidad_od.csv")
poblacion_total_zonas = resumen_masas["poblacion_total"].sum()
conexiones_fuertes = cfg_bb["garantia"]["conexiones_fuertes"]

nodos = [n["id"] for n in cfg_bb["nodos_demanda"]]
matriz_tiempos = pd.read_csv(BB_DIR / "02_ruteo_navegable" / "output" / "matriz_tiempos_min.csv", index_col=0)

cfg_agente = cfg_sim["agente"]
cfg_hiper = cfg_agente["hiperparametros"]
cfg_entren = cfg_agente["entrenamiento"]
cfg_escalon = cfg_sim["escalones"][cfg_entren["escalon_base"]]
hora_ini_min, hora_fin_min = cfg_escalon["horas"][0] * 60, cfg_escalon["horas"][1] * 60

# Overrides de recompensa SOLO para el entorno de entrenamiento/evaluacion RL --
# `recompensa:` del config no se toca (sigue gobernando escalones 1-3 y la politica
# base tal cual). Este es el combo GANADOR de 02_secuencia_experimentos_reward.ipynb.
cfg_recompensa_rl = dict(cfg_sim["recompensa"])
cfg_recompensa_rl.update(cfg_entren.get("recompensa_overrides", {}))

OUT_DIR = BASE_DIR / "output" / "modelo_final"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def construir_entorno_entrenamiento(semilla_entrenamiento):
    return EntornoDemandaAleatoria(
        generar_llegadas_dia_fn=demand_llegadas.generar_llegadas_dia,
        cfg_demand=cfg_demand, intensidad_od=intensidad_od, conexiones_fuertes=conexiones_fuertes,
        poblacion_total_zonas=poblacion_total_zonas, horas=cfg_escalon["horas"],
        porcentaje_poblacion_dia=cfg_escalon["porcentaje_poblacion_dia"],
        semilla_entrenamiento=semilla_entrenamiento,
        matriz_tiempos=matriz_tiempos, nodos=nodos, num_barcos=cfg_escalon["num_barcos"],
        capacidad_barco=cfg_bb["flota"]["capacidad_pasajeros"], nodo_inicial=cfg_bb["flota"]["nodo_inicial"],
        paso_tiempo_min=cfg_sim["paso_tiempo_min"], hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        cfg_recompensa=cfg_recompensa_rl, unidad_demanda=cfg_sim["unidad_demanda"],
    )


print(f"Instancia de entrenamiento: {cfg_entren['escalon_base']} "
      f"({cfg_escalon['num_barcos']} barcos, {cfg_escalon['horas']}h, "
      f"{cfg_escalon['porcentaje_poblacion_dia']*100:.1f}% poblacion/dia)")
print(f"Semilla de entrenamiento (raiz de la secuencia de demandas): {cfg_entren['semilla_entrenamiento']}")
print(f"Total timesteps: {cfg_entren['total_timesteps']}")
print(f"Hiperparametros PPO (compartidos con los experimentos): {cfg_hiper}")
print(f"Overrides de recompensa (combo ganador): {cfg_entren.get('recompensa_overrides', {})}")


Instancia de entrenamiento: escalon_1 (2 barcos, [6, 9]h, 0.8% poblacion/dia)
Semilla de entrenamiento (raiz de la secuencia de demandas): 123
Total timesteps: 150000
Hiperparametros PPO (compartidos con los experimentos): {'usar_vecnormalize': True, 'ent_coef': 0.01, 'learning_rate': 0.0003, 'n_steps': 512}
Overrides de recompensa (combo ganador): {'peso_movimiento': 0.003, 'penalizacion_maxima_persona': 1000000.0, 'premio_por_persona_entregada': 0.0}


## Confirmar demanda fresca por reset (antes de entrenar)

Dos `reset()` sin semilla deben dar demandas distintas; dos `reset(seed=42)`
deben dar la misma -- el mecanismo que hace posible el entrenamiento
Montecarlo (ver `entrenamiento.py`).

In [2]:
env_check = construir_entorno_entrenamiento(semilla_entrenamiento=cfg_entren["semilla_entrenamiento"])

env_check.reset(seed=42)
n_a = len(env_check.grupos_df)
env_check.reset(seed=42)
n_b = len(env_check.grupos_df)
env_check.reset()
n_c = len(env_check.grupos_df)
env_check.reset()
n_d = len(env_check.grupos_df)

print(f"reset(seed=42) dos veces: {n_a} y {n_b} grupos -- iguales: {n_a == n_b}")
print(f"reset() sin semilla dos veces: {n_c} y {n_d} grupos -- distintos: {n_c != n_d}")
assert n_a == n_b, "reset(seed=X) deberia ser reproducible"
assert n_c != n_d, "reset() sin semilla deberia dar demandas distintas"
print("\nOK: demanda fresca por reset confirmada.")


reset(seed=42) dos veces: 23 y 23 grupos -- iguales: True
reset() sin semilla dos veces: 20 y 25 grupos -- distintos: True

OK: demanda fresca por reset confirmada.


## `check_env` (chequeo estandar de SB3, antes de entrenar)

In [3]:
from stable_baselines3.common.env_checker import check_env

env_para_chequear = construir_entorno_entrenamiento(semilla_entrenamiento=cfg_entren["semilla_entrenamiento"])
check_env(env_para_chequear, warn=True)
print("OK: check_env no encontro problemas.")


OK: check_env no encontro problemas.


## Entrenar PPO

`Monitor` envuelve el entorno para registrar recompensa/duracion por episodio en un CSV (`output/rl_ppo/monitor.monitor.csv`), sin depender de TensorBoard -- se grafica mas abajo. Si `agente.entrenamiento.usar_vecnormalize` esta activo (config), el entorno tambien se envuelve en `DummyVecEnv` + `VecNormalize` (normaliza observaciones y recompensa con estadisticas corridas) -- ver `simulacion/README.md`, seccion de diagnostico, para por que hizo falta: sin esto, el primer intento de entrenamiento aprendio una politica degenerada (0% atendidas incluso en la instancia de juguete de verificacion).

In [4]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

usar_vecnormalize = cfg_hiper.get("usar_vecnormalize", False)

if usar_vecnormalize:
    venv = DummyVecEnv([lambda: Monitor(
        construir_entorno_entrenamiento(semilla_entrenamiento=cfg_entren["semilla_entrenamiento"]),
        filename=str(OUT_DIR / "monitor"),
    )])
    venv = VecNormalize(venv, norm_obs=True, norm_reward=True, gamma=cfg_agente["gamma"])
    env_para_ppo = venv
else:
    venv = None
    env_para_ppo = Monitor(
        construir_entorno_entrenamiento(semilla_entrenamiento=cfg_entren["semilla_entrenamiento"]),
        filename=str(OUT_DIR / "monitor"),
    )

model = PPO(
    "MlpPolicy", env_para_ppo,
    gamma=cfg_agente["gamma"],
    ent_coef=cfg_hiper.get("ent_coef", 0.0),
    learning_rate=cfg_hiper.get("learning_rate", 3e-4),
    n_steps=cfg_hiper.get("n_steps", 2048),
    seed=cfg_entren["semilla_entrenamiento"],
    verbose=1,
)
print(f"usar_vecnormalize={usar_vecnormalize}, ent_coef={model.ent_coef}, "
      f"learning_rate={model.learning_rate}, n_steps={model.n_steps}")
model.learn(total_timesteps=cfg_entren["total_timesteps"])


Using cpu device


usar_vecnormalize=True, ent_coef=0.01, learning_rate=0.0003, n_steps=512


----------------------------------
| rollout/           |           |
|    ep_len_mean     | 90        |
|    ep_rew_mean     | -2.29e+04 |
| time/              |           |
|    fps             | 394       |
|    iterations      | 1         |
|    time_elapsed    | 1         |
|    total_timesteps | 512       |
----------------------------------


----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 90         |
|    ep_rew_mean          | -2.53e+04  |
| time/                   |            |
|    fps                  | 358        |
|    iterations           | 2          |
|    time_elapsed         | 2          |
|    total_timesteps      | 1024       |
| train/                  |            |
|    approx_kl            | 0.00516599 |
|    clip_fraction        | 0.0139     |
|    clip_range           | 0.2        |
|    entropy_loss         | -3.22      |
|    explained_variance   | -0.0487    |
|    learning_rate        | 0.0003     |
|    loss                 | 4.17       |
|    n_updates            | 10         |
|    policy_gradient_loss | -0.0243    |
|    value_loss           | 12.9       |
----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 90          |
|    ep_rew_mean          | -2.59e+04   |
| time/                   |             |
|    fps                  | 360         |
|    iterations           | 3           |
|    time_elapsed         | 4           |
|    total_timesteps      | 1536        |
| train/                  |             |
|    approx_kl            | 0.011042165 |
|    clip_fraction        | 0.0623      |
|    clip_range           | 0.2         |
|    entropy_loss         | -3.21       |
|    explained_variance   | 0.258       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.0141      |
|    n_updates            | 20          |
|    policy_gradient_loss | -0.039      |
|    value_loss           | 0.792       |
-----------------------------------------


...

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 90          |
|    ep_rew_mean          | -3.67e+03   |
| time/                   |             |
|    fps                  | 242         |
|    iterations           | 289         |
|    time_elapsed         | 610         |
|    total_timesteps      | 147968      |
| train/                  |             |
|    approx_kl            | 0.018173708 |
|    clip_fraction        | 0.178       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.56       |
|    explained_variance   | 0.675       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0997     |
|    n_updates            | 2880        |
|    policy_gradient_loss | -0.0571     |
|    value_loss           | 0.0123      |
-----------------------------------------


----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 90         |
|    ep_rew_mean          | -3.54e+03  |
| time/                   |            |
|    fps                  | 242        |
|    iterations           | 290        |
|    time_elapsed         | 612        |
|    total_timesteps      | 148480     |
| train/                  |            |
|    approx_kl            | 0.02413463 |
|    clip_fraction        | 0.205      |
|    clip_range           | 0.2        |
|    entropy_loss         | -1.56      |
|    explained_variance   | 0.656      |
|    learning_rate        | 0.0003     |
|    loss                 | -0.107     |
|    n_updates            | 2890       |
|    policy_gradient_loss | -0.0604    |
|    value_loss           | 0.0215     |
----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 90          |
|    ep_rew_mean          | -3.35e+03   |
| time/                   |             |
|    fps                  | 242         |
|    iterations           | 291         |
|    time_elapsed         | 615         |
|    total_timesteps      | 148992      |
| train/                  |             |
|    approx_kl            | 0.022628568 |
|    clip_fraction        | 0.166       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.57       |
|    explained_variance   | 0.62        |
|    learning_rate        | 0.0003      |
|    loss                 | -0.106      |
|    n_updates            | 2900        |
|    policy_gradient_loss | -0.053      |
|    value_loss           | 0.0128      |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 90          |
|    ep_rew_mean          | -3.45e+03   |
| time/                   |             |
|    fps                  | 242         |
|    iterations           | 292         |
|    time_elapsed         | 617         |
|    total_timesteps      | 149504      |
| train/                  |             |
|    approx_kl            | 0.018103287 |
|    clip_fraction        | 0.169       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.6        |
|    explained_variance   | 0.797       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.097      |
|    n_updates            | 2910        |
|    policy_gradient_loss | -0.0537     |
|    value_loss           | 0.00859     |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 90          |
|    ep_rew_mean          | -3.47e+03   |
| time/                   |             |
|    fps                  | 242         |
|    iterations           | 293         |
|    time_elapsed         | 619         |
|    total_timesteps      | 150016      |
| train/                  |             |
|    approx_kl            | 0.021428686 |
|    clip_fraction        | 0.179       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.54       |
|    explained_variance   | 0.746       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.104      |
|    n_updates            | 2920        |
|    policy_gradient_loss | -0.0545     |
|    value_loss           | 0.0125      |
-----------------------------------------


## Guardar el modelo y graficar la curva de recompensa por episodio

In [5]:
model.save(str(OUT_DIR / "modelo_ppo"))
if venv is not None:
    venv.save(str(OUT_DIR / "vecnormalize.pkl"))
    print("Modelo y estadisticas de VecNormalize guardados en", OUT_DIR)
else:
    print("Modelo guardado en", OUT_DIR / "modelo_ppo.zip")


Modelo y estadisticas de VecNormalize guardados en C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\modelo_rl\output\modelo_final


In [6]:
import plotly.graph_objects as go

# Se lee el CSV del monitor DIRECTAMENTE (no con `load_results(OUT_DIR)`, que junta
# TODOS los *.monitor.csv de la carpeta -- mezclaria esta corrida con la de
# 04a_verificacion_toy_rl.ipynb, que guarda la suya en el mismo OUT_DIR).
monitor_path = OUT_DIR / "monitor.monitor.csv"
df_monitor = pd.read_csv(monitor_path, skiprows=1)
y = df_monitor["r"].values
x = list(range(len(y)))

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode="lines", line=dict(width=1, color="rgba(41,128,185,0.35)"), name="episode reward"))
if len(y) >= 20:
    media_movil = pd.Series(y).rolling(20, min_periods=1).mean()
    fig.add_trace(go.Scatter(x=x, y=media_movil, mode="lines", line=dict(width=2.5, color="#2980b9"), name="rolling mean (20 episodes)"))
fig.update_layout(
    title=f"Training curve -- PPO on {cfg_entren['escalon_base']}",
    xaxis_title="Episode", yaxis_title="Episode total reward",
    template="plotly_white",
)
fig.write_html(OUT_DIR / "training_curve.html")
fig.show()

print(f"{len(x)} episodes trained. Reward -- first 20: {y[:20].mean():.2f}, last 20: {y[-20:].mean():.2f}")


1666 episodes trained. Reward -- first 20: -24871.44, last 20: -2832.81


## Prueba aparte: `n_steps=90` en vez de 512

512 (el valor usado arriba) no sale de una formula ni de una referencia -- es
mas chico que el default de Stable-Baselines3 (2048) porque con episodios de
90 pasos, actualizar cada ~22 episodios parecia demasiado lento, pero 512 en
si mismo fue una eleccion de una ronda de diagnostico anterior, no algo
derivado del problema. La alternativa mas directa de justificar es
`n_steps=90`: exactamente la duracion de un episodio completo en
`escalon_1`, asi que PPO actualiza la red justo al cerrar cada episodio,
nunca a mitad de uno -- cada rollout es un numero entero de episodios
completos.

Se entrena de nuevo, mismo combo de recompensa (C1), mismos 150000
timesteps, unico cambio: `n_steps`. Se guarda en una carpeta aparte
(`output/modelo_final_n90/`) -- no se pisa el modelo de arriba, se comparan
los dos en `comparacion/`.

In [7]:
cfg_entren_n90 = cfg_agente["entrenamiento_n90"]
cfg_recompensa_n90 = dict(cfg_sim["recompensa"])
cfg_recompensa_n90.update(cfg_entren_n90.get("recompensa_overrides", {}))

OUT_DIR_N90 = BASE_DIR / "output" / "modelo_final_n90"
OUT_DIR_N90.mkdir(parents=True, exist_ok=True)


def construir_entorno_n90(semilla_entrenamiento):
    return EntornoDemandaAleatoria(
        generar_llegadas_dia_fn=demand_llegadas.generar_llegadas_dia,
        cfg_demand=cfg_demand, intensidad_od=intensidad_od, conexiones_fuertes=conexiones_fuertes,
        poblacion_total_zonas=poblacion_total_zonas, horas=cfg_escalon["horas"],
        porcentaje_poblacion_dia=cfg_escalon["porcentaje_poblacion_dia"],
        semilla_entrenamiento=semilla_entrenamiento,
        matriz_tiempos=matriz_tiempos, nodos=nodos, num_barcos=cfg_escalon["num_barcos"],
        capacidad_barco=cfg_bb["flota"]["capacidad_pasajeros"], nodo_inicial=cfg_bb["flota"]["nodo_inicial"],
        paso_tiempo_min=cfg_sim["paso_tiempo_min"], hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        cfg_recompensa=cfg_recompensa_n90, unidad_demanda=cfg_sim["unidad_demanda"],
    )


venv_n90 = DummyVecEnv([lambda: Monitor(
    construir_entorno_n90(semilla_entrenamiento=cfg_entren_n90["semilla_entrenamiento"]),
    filename=str(OUT_DIR_N90 / "monitor"),
)])
venv_n90 = VecNormalize(venv_n90, norm_obs=True, norm_reward=True, gamma=cfg_agente["gamma"])

model_n90 = PPO(
    "MlpPolicy", venv_n90,
    gamma=cfg_agente["gamma"],
    ent_coef=cfg_hiper.get("ent_coef", 0.0),
    learning_rate=cfg_hiper.get("learning_rate", 3e-4),
    n_steps=cfg_entren_n90["n_steps"],
    seed=cfg_entren_n90["semilla_entrenamiento"],
    verbose=1,
)
print(f"n_steps={model_n90.n_steps} (vs. {cfg_hiper['n_steps']} de la corrida de arriba), "
      f"ent_coef={model_n90.ent_coef}, learning_rate={model_n90.learning_rate}")
model_n90.learn(total_timesteps=cfg_entren_n90["total_timesteps"])


Using cpu device
n_steps=90 (vs. 512 de la corrida de arriba), ent_coef=0.01, learning_rate=0.0003


C:\Users\hfons\AppData\Local\Programs\Python\Python314\Lib\site-packages\stable_baselines3\ppo\ppo.py:155: UserWarning: You have specified a mini-batch size of 64, but because the `RolloutBuffer` is of size `n_steps * n_envs = 90`, after every 1 untruncated mini-batches, there will be a truncated mini-batch of size 26
We recommend using a `batch_size` that is a factor of `n_steps * n_envs`.
Info: (n_steps=90 and n_envs=1)
  warnings.warn(


----------------------------------
| rollout/           |           |
|    ep_len_mean     | 90        |
|    ep_rew_mean     | -1.31e+04 |
| time/              |           |
|    fps             | 235       |
|    iterations      | 1         |
|    time_elapsed    | 0         |
|    total_timesteps | 90        |
----------------------------------


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 90           |
|    ep_rew_mean          | -1.45e+04    |
| time/                   |              |
|    fps                  | 209          |
|    iterations           | 2            |
|    time_elapsed         | 0            |
|    total_timesteps      | 180          |
| train/                  |              |
|    approx_kl            | 0.0030489143 |
|    clip_fraction        | 0.00192      |
|    clip_range           | 0.2          |
|    entropy_loss         | -3.22        |
|    explained_variance   | 0.0271       |
|    learning_rate        | 0.0003       |
|    loss                 | 38           |
|    n_updates            | 10           |
|    policy_gradient_loss | -0.0284      |
|    value_loss           | 75.1         |
------------------------------------------


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 90           |
|    ep_rew_mean          | -1.43e+04    |
| time/                   |              |
|    fps                  | 200          |
|    iterations           | 3            |
|    time_elapsed         | 1            |
|    total_timesteps      | 270          |
| train/                  |              |
|    approx_kl            | 0.0048899613 |
|    clip_fraction        | 0.01         |
|    clip_range           | 0.2          |
|    entropy_loss         | -3.22        |
|    explained_variance   | -0.642       |
|    learning_rate        | 0.0003       |
|    loss                 | 0.474        |
|    n_updates            | 20           |
|    policy_gradient_loss | -0.0276      |
|    value_loss           | 1.48         |
------------------------------------------


...

----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 90         |
|    ep_rew_mean          | -3.81e+03  |
| time/                   |            |
|    fps                  | 197        |
|    iterations           | 1663       |
|    time_elapsed         | 756        |
|    total_timesteps      | 149670     |
| train/                  |            |
|    approx_kl            | 0.03597109 |
|    clip_fraction        | 0.159      |
|    clip_range           | 0.2        |
|    entropy_loss         | -1.01      |
|    explained_variance   | 0.706      |
|    learning_rate        | 0.0003     |
|    loss                 | -0.0873    |
|    n_updates            | 16620      |
|    policy_gradient_loss | -0.0492    |
|    value_loss           | 0.00823    |
----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 90          |
|    ep_rew_mean          | -3.71e+03   |
| time/                   |             |
|    fps                  | 198         |
|    iterations           | 1664        |
|    time_elapsed         | 756         |
|    total_timesteps      | 149760      |
| train/                  |             |
|    approx_kl            | 0.020771159 |
|    clip_fraction        | 0.109       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.922      |
|    explained_variance   | 0.814       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.102      |
|    n_updates            | 16630       |
|    policy_gradient_loss | -0.0323     |
|    value_loss           | 0.0204      |
-----------------------------------------


----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 90         |
|    ep_rew_mean          | -3.55e+03  |
| time/                   |            |
|    fps                  | 198        |
|    iterations           | 1665       |
|    time_elapsed         | 756        |
|    total_timesteps      | 149850     |
| train/                  |            |
|    approx_kl            | 0.01185946 |
|    clip_fraction        | 0.0605     |
|    clip_range           | 0.2        |
|    entropy_loss         | -1.03      |
|    explained_variance   | 0.771      |
|    learning_rate        | 0.0003     |
|    loss                 | -0.083     |
|    n_updates            | 16640      |
|    policy_gradient_loss | -0.0306    |
|    value_loss           | 0.00755    |
----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 90          |
|    ep_rew_mean          | -3.54e+03   |
| time/                   |             |
|    fps                  | 198         |
|    iterations           | 1666        |
|    time_elapsed         | 756         |
|    total_timesteps      | 149940      |
| train/                  |             |
|    approx_kl            | 0.014965288 |
|    clip_fraction        | 0.0822      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.15       |
|    explained_variance   | 0.831       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0639     |
|    n_updates            | 16650       |
|    policy_gradient_loss | -0.0338     |
|    value_loss           | 0.00984     |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 90          |
|    ep_rew_mean          | -3.56e+03   |
| time/                   |             |
|    fps                  | 198         |
|    iterations           | 1667        |
|    time_elapsed         | 757         |
|    total_timesteps      | 150030      |
| train/                  |             |
|    approx_kl            | 0.012909262 |
|    clip_fraction        | 0.0793      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.1        |
|    explained_variance   | 0.709       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0708     |
|    n_updates            | 16660       |
|    policy_gradient_loss | -0.034      |
|    value_loss           | 0.00526     |
-----------------------------------------


In [8]:
model_n90.save(str(OUT_DIR_N90 / "modelo_ppo"))
venv_n90.save(str(OUT_DIR_N90 / "vecnormalize.pkl"))
print("Modelo n_steps=90 guardado en", OUT_DIR_N90)


Modelo n_steps=90 guardado en C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\modelo_rl\output\modelo_final_n90


In [9]:
df_monitor_n90 = pd.read_csv(OUT_DIR_N90 / "monitor.monitor.csv", skiprows=1)
y90 = df_monitor_n90["r"].values

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(y))), y=pd.Series(y).rolling(20, min_periods=1).mean(),
                          mode="lines", line=dict(width=2, color="#2980b9"), name="n_steps=512 (rolling mean 20 ep)"))
fig.add_trace(go.Scatter(x=list(range(len(y90))), y=pd.Series(y90).rolling(20, min_periods=1).mean(),
                          mode="lines", line=dict(width=2, color="#e67e22"), name="n_steps=90 (rolling mean 20 ep)"))
fig.update_layout(title="Curva de entrenamiento -- n_steps=512 vs n_steps=90", xaxis_title="Episodio",
                   yaxis_title="Recompensa por episodio (media movil 20 ep)", template="plotly_white")
fig.write_html(str(OUT_DIR_N90 / "comparacion_curvas_n_steps.html"))
fig.show()

print(f"n_steps=512 -- primeros 20 ep: {y[:20].mean():.1f}, ultimos 20 ep: {y[-20:].mean():.1f}")
print(f"n_steps=90  -- primeros 20 ep: {y90[:20].mean():.1f}, ultimos 20 ep: {y90[-20:].mean():.1f}")


n_steps=512 -- primeros 20 ep: -24871.4, ultimos 20 ep: -2832.8
n_steps=90  -- primeros 20 ep: -22513.4, ultimos 20 ep: -2956.2
